# ?? Task 3: TalentMatch ML ? Resume & Candidate Screening System
### Track: Machine Learning (`ML`) | Internship ID: `FIT/AUG26/ML10465` | Repository: `FUTURE_ML_03`

---
### **Project Overview & Objectives**
In modern technical recruitment, manual resume review is prone to human bias, keyword-stuffing exploits, and inconsistent evaluations across candidates.

**TalentMatch ML** is a fair, explainable Machine Learning screening engine that:
1. Parses multi-format candidate documents (`.pdf`, `.docx`, `.txt`).
2. Anonymizes Personally Identifiable Information (PII) to eliminate demographic bias.
3. Extracts technical competencies using a curated 200+ skill taxonomy with alias resolution.
4. Performs **multi-tier hybrid matching**:
   - **Tier 1**: Structured Taxonomy Overlap (Mandatory vs. Preferred Skills).
   - **Tier 2**: Dense Semantic Similarity via `SentenceTransformer` (`all-MiniLM-L6-v2`).
   - **Tier 3**: Lexical TF-IDF Cosine Similarity.
   - **Tier 4**: Experience & Education Alignment.
5. Ranks candidates and produces detailed **Skill Gap Diagnostics** (highlighting matched and missing competencies).


In [1]:
import os
import sys
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

sys.path.append(os.path.abspath('..'))

from src.data_generator import generate_sample_resumes, generate_sample_job_descriptions
from src.parser import parse_resume
from src.matcher import TalentMatcher
from src.visualize import plot_candidate_leaderboard, plot_score_component_breakdown

print("? TalentMatch ML environment initialized. Python:", sys.version.split()[0])

? TalentMatch ML environment initialized. Python: 3.12.5


## 1. Document Ingestion & PII Anonymization

In [2]:
resumes_dir = os.path.join('..', 'data', 'resumes')
jds_dir = os.path.join('..', 'data', 'job_descriptions')

generate_sample_resumes(resumes_dir)
generate_sample_job_descriptions(jds_dir)

resume_files = sorted(glob.glob(os.path.join(resumes_dir, '*.*')))
print(f"Total Resumes Loaded: {len(resume_files)}")

# Parse & Inspect sample anonymization
sample_cand = parse_resume(resume_files[0])
print("Candidate ID:", sample_cand["candidate_id"])
print("Experience:", sample_cand["experience_years"], "years | Education:", sample_cand["education_level"])
print("\nAnonymized Text Preview:\n", sample_cand["anonymized_text"][:350], "...")

[DataGenerator] 8 sample resumes created across DOCX & TXT formats in ..\data\resumes
[DataGenerator] 3 structured Job Descriptions saved to ..\data\job_descriptions
Total Resumes Loaded: 8
Candidate ID: candidate_01_lead_ml_engineer
Experience: 6.5 years | Education: Master's Degree

Anonymized Text Preview:
 Alex Rivera — Senior Machine Learning & MLOps Engineer
Email: [EMAIL_MASKED] | Phone: [PHONE_MASKED] | GitHub: github.com/arivera-ml
Professional Summary
Senior Machine Learning Engineer with 6.5 years of experience building, training, and deploying large-scale deep learning models, LLM pipelines, and automated MLOps infrastructure in cloud environ ...


## 2. Job Description Specification & Technical Taxonomy

In [3]:
with open(os.path.join(jds_dir, 'job_senior_ml_engineer.json'), 'r') as f:
    target_jd = json.load(f)

print("?? Target Job Role:", target_jd["title"])
print("Minimum Experience:", target_jd["min_experience_years"], "Years")
print("\nMandatory Skills (8):", target_jd["required_skills"])
print("Preferred Skills (6):", target_jd["preferred_skills"])

?? Target Job Role: Senior Machine Learning Engineer (NLP & MLOps)
Minimum Experience: 4.0 Years

Mandatory Skills (8): ['python', 'pytorch', 'scikit-learn', 'machine learning', 'natural language processing', 'docker', 'aws', 'model deployment']
Preferred Skills (6): ['transformers', 'huggingface', 'kubernetes', 'mlops', 'langchain', 'sql']


## 3. Hybrid Candidate Matching & Ranking

In [4]:
matcher = TalentMatcher(use_dense_embeddings=True)
candidate_profiles = [parse_resume(f) for f in resume_files]

rankings_df = matcher.rank_candidates(candidate_profiles, target_jd)
display(rankings_df[[
    "Rank", "candidate_id", "composite_score", "gap_severity",
    "skill_score_pct", "semantic_similarity_pct", "lexical_similarity_pct", "experience_years"
]].style.highlight_max(subset=["composite_score"], color="#c7e9c0"))

C:\Users\gonna\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[TalentMatcher] Loaded SentenceTransformer (all-MiniLM-L6-v2) for dense semantic scoring.


,Rank,candidate_id,composite_score,gap_severity,skill_score_pct,semantic_similarity_pct,lexical_similarity_pct,experience_years
0,1,candidate_01_lead_ml_engineer,77.900000,Low,100.000000,78.300000,22.000000,6.500000
1,2,candidate_02_data_scientist,40.500000,Moderate,43.300000,39.600000,6.500000,4.000000
2,3,candidate_06_devops_cloud_architect,39.980000,High,33.300000,51.000000,6.700000,8.000000
3,4,candidate_03_junior_ml_intern,38.500000,Moderate,40.000000,50.800000,11.300000,2.000000
4,5,candidate_04_fullstack_lead,33.250000,High,20.000000,48.200000,4.000000,7.000000
5,6,candidate_08_java_backend_dev,28.740000,High,13.300000,42.400000,3.400000,5.500000
6,7,candidate_07_bi_data_analyst,25.790000,High,13.300000,33.900000,1.500000,4.500000
7,8,candidate_05_frontend_dev,17.760000,High,0.000000,33.000000,1.800000,3.000000


## 4. Visual Diagnostics & Score Decomposition

In [5]:
plt.figure(figsize=(12, 5))
df_plot = rankings_df.sort_values(by="composite_score", ascending=True)
colors = ["#2ca02c" if s >= 75 else ("#ff7f0e" if s >= 50 else "#d62728") for s in df_plot["composite_score"]]
plt.barh(df_plot["candidate_id"], df_plot["composite_score"], color=colors, edgecolor="#333", height=0.55)
plt.axvline(75, color="#2ca02c", linestyle="--", label="Strong Match (75%)")
plt.title(f"TalentMatch ML Leaderboard ? {target_jd['title']}", fontsize=12, fontweight="bold")
plt.xlabel("Composite Match Score (%)")
plt.legend()
plt.show()

C:\Users\gonna\AppData\Local\Temp\ipykernel_11764\3938244394.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Candidate Deep-Dive & Skill-Gap Analysis

In [6]:
for idx, row in rankings_df.head(4).iterrows():
    print("=" * 80)
    print(f"Rank {row['Rank']}: Candidate '{row['candidate_id']}' (Composite Score: {row['composite_score']:.1f}%)")
    print(f"  ? Matched Mandatory: {row['matched_required']}")
    print(f"  ? Missing Mandatory: {row['missing_required']}")
    print(f"  ? Matched Preferred: {row['matched_preferred']}")
    print(f"  ? Gap Severity:      {row['gap_severity']}")
    print(f"  ? Recommendation:    {row['recommendation']}")

Rank 1: Candidate 'candidate_01_lead_ml_engineer' (Composite Score: 77.9%)
  ? Matched Mandatory: ['amazon web services', 'docker', 'machine learning', 'model deployment', 'natural language processing', 'python', 'pytorch', 'scikit-learn']
  ? Missing Mandatory: []
  ? Matched Preferred: ['huggingface', 'kubernetes', 'langchain', 'mlops', 'sql', 'transformers']
  ? Gap Severity:      Low
  ? Recommendation:    Potentially Qualified: Moderate skill match. Review missing skills with hiring manager before screen.
Rank 2: Candidate 'candidate_02_data_scientist' (Composite Score: 40.5%)
  ? Matched Mandatory: ['amazon web services', 'machine learning', 'python', 'scikit-learn']
  ? Missing Mandatory: ['docker', 'model deployment', 'natural language processing', 'pytorch']
  ? Matched Preferred: ['sql']
  ? Gap Severity:      Moderate
  ? Recommendation:    Skill Gap Identified: Candidate lacks several mandatory technical competencies for this role.
Rank 3: Candidate 'candidate_06_devops_clo